In [27]:
# correct working directory only once 
if not "working_directory_corrected" in vars():
    %cd ..
    working_directory_corrected = True

In [34]:
import os
from sklearn.model_selection import train_test_split
import numpy as np
from utils.true_distance.generate_true_distance_dataset import generate_true_distance_dataset
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
import pandas as pd 


import time
import random
from stable_worldmodel.envs.two_room import TwoRoomEnv
from utils.true_distance.generate_true_distance_dataset import (
    build_occupancy_grid,
    dijkstra_distance,
    true_distance_batch,
    _wall_geometry,
    _sample_valid_position,
)

### Dataset with 1000 and 10000 samples

In this section we create 2 datasets; one with 1000 samples and another one with 10000 samples. We save both locally in our /data folder as a pickle file. 

In [29]:
SAMPLE_SIZES = [1000, 10000]
datasets = {}

for n in SAMPLE_SIZES:
    path = f"data/true_distance_heuristic_dataset_{n}.pkl"
    if os.path.exists(path):
        datasets[n] = pd.read_pickle(path)
    else:
        datasets[n] = generate_true_distance_dataset(num_samples=n, seed=n)
        datasets[n].to_pickle(path)

### Model training

80/20 train/test split with fixed model parameters 

In [30]:
def heuristic_dataset_to_xy(df):
    agent_encs = np.array(df['agent_pos_enc'].tolist(), dtype=float)
    goal_encs = np.array(df['goal_pos_enc'].tolist(), dtype=float)
    X = np.hstack([agent_encs, goal_encs])
    y = df['heuristic'].values.astype(float)
    return X, y

def train_and_evaluate(X, y, test_size=0.2, random_state=42):
    x_train, x_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
        "MLP Neural Net": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
    }

    results = {}
    for name, model in models.items():
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)
        results[name] = {
            "MAE": mean_absolute_error(y_test, y_pred),
            "MSE": mean_squared_error(y_test, y_pred),
            "R2": r2_score(y_test, y_pred),
            "Pearson": pearsonr(y_test, y_pred)[0],
            "Spearman": spearmanr(y_test, y_pred)[0],
        }

    return pd.DataFrame(results).T

In [31]:
results = {}

for n, df in datasets.items():
    X, y = heuristic_dataset_to_xy(df)
    results[n] = train_and_evaluate(X, y)

### Comparison: 1000 vs 10000 samples

In this section we want to compare our evaluation matrics for all models trained on 1000 vs 10000 samples. This should give us a first clue if we should increase the set size

In [33]:
comparison = pd.concat({f"{n} samples": r for n, r in results.items()})
comparison

MAE           MSE        R2   Pearson  \
1000 samples  Linear Regression  96.159764  23003.355819 -3.586360  0.108705   
              Random Forest      58.476334   5021.872705 -0.001250  0.144834   
              MLP Neural Net     59.144309   5014.515877  0.000217  0.203953   
10000 samples Linear Regression  56.930704   4547.162349  0.126580  0.363024   
              Random Forest      56.832543   4670.717433  0.102847  0.320741   
              MLP Neural Net     60.350915   5157.053353  0.009432  0.110057   

                                 Spearman  
1000 samples  Linear Regression  0.122357  
              Random Forest      0.087871  
              MLP Neural Net     0.168870  
10000 samples Linear Regression  0.343720  
              Random Forest      0.290894  
              MLP Neural Net     0.072965

Linear Regression and Random Forest clearly benefit from more data (R2 and correlations roughly triple going from 1000 to 10000 samples). MLP barely improves and even loses a bit of rank correlation

### Dijkstra vs. closed-form formula

We replaced the grid-search (Dijkstra) distance with a vectorized closed-form formula specified for the exact maze config on hand. Here we check both correctness and speed, timed over 50 pairs split into 10 batches of 5 to validate if this new approach works properly and fast 

In [35]:
env = TwoRoomEnv()
env.reset()
walkable = build_occupancy_grid(env)
geom = _wall_geometry(env)
rng = random.Random(0)

NUM_BATCHES = 10
BATCH_SIZE = 5

# pre-sample all 50 pairs upfront so both methods are timed on the exact same data
batches = []
for _ in range(NUM_BATCHES):
    pos_a = [_sample_valid_position(env, walkable, rng) for _ in range(BATCH_SIZE)]
    pos_g = [_sample_valid_position(env, walkable, rng) for _ in range(BATCH_SIZE)]
    batches.append((pos_a, pos_g))

In [36]:
timing_rows = []
max_abs_diff = 0.0

for batch_idx, (pos_a, pos_g) in enumerate(batches):
    t0 = time.perf_counter()
    dijkstra_results = [dijkstra_distance(walkable, a, g) for a, g in zip(pos_a, pos_g)]
    t_dijkstra = time.perf_counter() - t0

    t0 = time.perf_counter()
    formula_results = true_distance_batch(pos_a, pos_g, geom)
    t_formula = time.perf_counter() - t0

    diffs = [abs(d - f) for d, f in zip(dijkstra_results, formula_results)]
    max_abs_diff = max(max_abs_diff, max(diffs))

    timing_rows.append({
        "batch": batch_idx,
        "dijkstra_s": t_dijkstra,
        "formula_s": t_formula,
        "speedup": t_dijkstra / t_formula,
    })

timing_df = pd.DataFrame(timing_rows)
timing_df

,batch,dijkstra_s,formula_s,speedup
0,0,0.220309,0.000108,2031.262126
1,1,0.195176,0.000090,2162.617767
2,2,0.144576,0.000077,1880.669305
3,3,0.093035,0.000083,1116.415853
4,4,0.089405,0.000055,1623.088791
5,5,0.144137,0.000063,2281.842804
6,6,0.210711,0.000108,1952.525575
7,7,0.133804,0.000085,1577.240576
8,8,0.088816,0.000077,1153.449612
9,9,0.234131,0.000081,2880.117202


In [37]:
print(f"Total Dijkstra time ({NUM_BATCHES * BATCH_SIZE} pairs): {timing_df['dijkstra_s'].sum():.3f}s")
print(f"Total formula time  ({NUM_BATCHES * BATCH_SIZE} pairs): {timing_df['formula_s'].sum()*1000:.3f}ms")
print(f"Average speedup: {timing_df['speedup'].mean():.0f}x")
print(f"Max absolute difference between Dijkstra and formula: {max_abs_diff:.3f} (grid-discretization margin)")

Total Dijkstra time (50 pairs): 1.554s
Total formula time  (50 pairs): 0.828ms
Average speedup: 1866x
Max absolute difference between Dijkstra and formula: 30.724 (grid-discretization margin)
